### Word2Vec

Word2Vec is a family of neural network models that learn dense vector representations of words from large text corpora. Instead of one-hot encoding, it maps each word to a continuous vector in a low-dimensional space where semantic and syntactic relationships are captured by distances and directions.

How it works:
- Two main architectures:
    - Skip-gram: predicts surrounding context words from a target word.
    - CBOW (Continuous Bag of Words): predicts the target word from surrounding context words.
- Training uses a shallow neural network with one hidden layer; the learned weights become the word embeddings.
- Optimizations like negative sampling or hierarchical softmax make training fast on large vocabularies.
- A context window defines how many words before/after the target word are considered.

Why it works:
- Words appearing in similar contexts get similar vectors, encoding distributional similarity.
- Vector arithmetic can capture analogies (e.g., `king - man + woman ≈ queen`).
- Embeddings are continuous, dense, and capture multiple degrees of similarity.

Pros:
- Efficient to train and use.
- Produces high-quality semantic embeddings.
- Works well with large unlabeled corpora.
- Embeddings can be reused for downstream NLP tasks.

Cons:
- Context-independent: each word has a single vector regardless of sense ambiguity.
- Requires large corpora for best results.
- Rare words and out-of-vocabulary words are poorly handled.
- Captures statistical co-occurrence, not true world knowledge or complex syntax.

In [74]:
import numpy as np
import pandas as pd

In [ ]:
import gensim 
import os

from nltk import sent_tokenize, word_tokenize
from gensim.utils import simple_preprocess

story = "Once upon a time, there was a little girl"
simple_preprocess(story)

['once', 'upon', 'time', 'there', 'was', 'little', 'girl']

In [76]:
story = []  # initialize list of lists of strings (tokenized words from sentences)

path = os.path.join('data', 'game-of-thrones')  # set the directory path
for file in os.listdir(path):  # loop over files in the directory
    file_path = os.path.join(path, file)  # build the full file path
    with open(file_path, 'r') as f:  # open the file in text mode
        corpus = f.read()  # read the file content
    
    raw_sentences = sent_tokenize(corpus)  # split the text into sentences
    for sentence in raw_sentences:  # iterate through each sentence
        # preprocess/tokenize and append to story
        story.append(simple_preprocess(sentence))

In [108]:
print("total tokenized sentences: ", len(story))

total tokenized sentences:  158874


In [78]:
model = gensim.models.Word2Vec(
    window=10,
    min_count=1,
    workers=4
    )

In [79]:
model.build_vocab(story)  # build the vocabulary from the tokenized sentences

In [115]:
model.corpus_count, model.corpus_total_words

(158874, 1725053)

In [86]:
model.epochs = 10

In [ ]:
model.train(
    story,  # train the model on the tokenized sentences
    total_examples=model.corpus_count,  # number of sentences
    epochs=model.epochs  # number of epochs
) # set sg=1 for skip-gram, default is cbow

(13220957, 17250530)

In [88]:
model.wv.most_similar('king')

[('kings', 0.8040199875831604),
 ('baratheon', 0.6794708967208862),
 ('realm', 0.6153817176818848),
 ('usurper', 0.5951004028320312),
 ('throne', 0.5763828754425049),
 ('victory', 0.5336558222770691),
 ('pagaentry', 0.5331382155418396),
 ('site', 0.5293996930122375),
 ('conqueror', 0.5267696976661682),
 ('dragonstone', 0.526432454586029)]

In [89]:
model.wv.most_similar('daenerys')

[('stormborn', 0.7291814684867859),
 ('targaryen', 0.6870346665382385),
 ('unburnt', 0.61597740650177),
 ('viserys', 0.5962873101234436),
 ('queen', 0.5850672721862793),
 ('aegon', 0.5770994424819946),
 ('myrcella', 0.5724384784698486),
 ('elia', 0.5656646490097046),
 ('princess', 0.5561906099319458),
 ('dragons', 0.5482234954833984)]

In [92]:
model.wv['king']  # get the vector for the word 'king' of size 100 (default)

array([ 3.541402  , -1.7346166 ,  1.1958182 , -1.9272329 , -3.885938  ,
        4.235655  ,  0.9691832 , -1.7045882 ,  3.5584724 ,  0.6660395 ,
       -1.241416  ,  1.4380672 , -2.1845357 , -0.15620774, -2.6526241 ,
       -1.013578  ,  0.51378566,  0.20161377, -4.494435  , -0.48997164,
        1.5548604 ,  3.595777  , -0.3769844 , -3.3884976 ,  0.57737494,
       -1.9661345 , -0.9182713 ,  2.3580246 ,  1.6318018 ,  3.1370378 ,
       -2.5916758 ,  1.4623417 ,  0.20792365, -1.26101   ,  3.5163786 ,
       -0.05023661,  1.2575524 , -1.5004057 , -3.571278  ,  0.96945864,
       -1.6592026 , -1.2831339 , -0.7581684 ,  2.3328247 , -0.18879935,
       -4.3469243 , -0.21986386,  2.3257349 ,  1.6009511 ,  1.8181181 ,
       -0.89149415, -1.5199918 ,  0.01005308, -1.4509192 , -1.0323881 ,
        1.8174783 ,  1.2852052 ,  1.7022581 ,  0.6836285 ,  1.1708995 ,
        3.3105168 ,  0.60916775,  2.2487824 ,  1.2829496 ,  0.52207524,
       -0.23850466, -0.3666589 ,  0.65656805, -1.5711454 , -1.30

In [ ]:
# find the word that doesn't match in the list
model.wv.doesnt_match(['king', 'lannister', 'kingsguard', 'jaime', 'daenerys'])

'daenerys'

In [104]:
model.wv.similarity('arya', 'sansa')

0.7964028

In [ ]:
# get the normalized vectors for all words in the vocabulary
model.wv.get_normed_vectors().shape

(24223, 100)